# Read V1DD soma features parquet

Reads and displays `soma_features.parquet` from the V1DD 1196 release.

In [1]:
from pathlib import Path
import os

import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema
from connects_common_connectivity.models import CellFeatureDefinition, CellFeatureMatrix, CellFeatureSet, Unit

In [4]:
DATA_ROOT = Path("/data/v1dd_v1196_soma_features")
SOMA_FEATURES_PATH = DATA_ROOT / "soma_features.parquet"

In [5]:
soma_features = pl.read_parquet(SOMA_FEATURES_PATH)
print(soma_features.shape)

(191117, 13)


In [6]:
soma_features.head()

soma_id,nucleus_id,nucleus_volume_um,nucleus_area_um,nuclear_area_to_volume_ratio,nuclear_folding_area_um,fraction_nuclear_folding,nucleus_to_soma_ratio,soma_volume_um,soma_area_um,soma_to_nucleus_center_dist,soma_area_to_volume_ratio,soma_synapse_density_um
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
864691132578185492,266003,69.066691,167.169546,2.053496,4.704542,0.028142,0.044117,1565.542207,6954.105542,6.315304,10.665486,0.007478
864691132604766232,389078,303.856672,320.451624,1.466108,52.221526,0.162962,0.424058,716.545592,811.875095,1.630088,2.096569,0.081293
864691132593478952,372838,246.926466,289.549431,1.521225,66.224893,0.228717,0.971701,254.117633,286.953025,0.157289,1.479007,0.0
864691132748929891,136884,271.945308,255.177943,1.257104,18.570041,0.072773,0.486058,559.491428,905.288395,0.876373,2.757024,0.047499
864691132561854219,677811,72.030333,146.136825,1.745549,37.69667,0.257955,1.33342,54.019229,110.19366,0.522126,1.594558,0.0


## Data validation cleanup

In [7]:
OUTPUT_ROOT = Path("../results/v1dd_1196_v3/")
PROJECT_ID  = "v1dd"
DENDRITE_DATASET_ID = "v1dd_1196_em"
FEATURE_SET_ID = "v1dd_soma_features"

In [8]:
assoc = (
    pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DENDRITE_DATASET_ID))
)
assert assoc.shape[0] > 0, f"No DataItems found for {DENDRITE_DATASET_ID}."

dendrite_ids = set(assoc["dataitem_id"].to_list())
soma_features_cc = (
    soma_features
    .with_columns(pl.col("soma_id").cast(pl.Utf8).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").is_in(list(dendrite_ids)))
    .unique(subset=["id"], keep="first")
)

assert soma_features_cc.height > 0, "No soma feature rows remained after dataset filter."
assert set(soma_features_cc["id"].to_list()).issubset(dendrite_ids)

print(f"registered rootids: {len(dendrite_ids)}")
print(f"soma features rows kept: {soma_features_cc.height}")

registered rootids: 143448
soma features rows kept: 121429


## Write the features

In [9]:
FEATURE_COLUMNS = [
    c for c in soma_features_cc.columns
    if c not in {"soma_id", "nucleus_id", "id"}
]

UNIT_BY_FEATURE = {
    "nucleus_volume_um": Unit.MICRONS_CUBED.value,
    "nucleus_area_um": Unit.MICRONS_SQUARE.value,
    "nuclear_area_to_volume_ratio": Unit.RATIO.value,
    "nuclear_folding_area_um": Unit.MICRONS_SQUARE.value,
    "fraction_nuclear_folding": Unit.RATIO.value,
    "nucleus_to_soma_ratio": Unit.RATIO.value,
    "soma_volume_um": Unit.MICRONS_CUBED.value,
    "soma_area_um": Unit.MICRONS_SQUARE.value,
    "soma_to_nucleus_center_dist": Unit.MICRONS_LENGTH.value,
    "soma_area_to_volume_ratio": Unit.RATIO.value,
    "soma_synapse_density_um": Unit.COUNT_PER_MICRONS_CUBED.value,
}

feature_defs = [
    CellFeatureDefinition(
        id=col,
        description=col.replace("_", " "),
        unit=UNIT_BY_FEATURE[col],
        data_type="<f8",
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    )
    for col in FEATURE_COLUMNS
]
write_models(feature_defs, output_root=OUTPUT_ROOT)

WrittenResult(class_name='CellFeatureDefinition', path=PosixPath('../results/v1dd_1196_v3/cellfeaturedefinition'), mode='overwrite_scoped', predicates=("project_id = 'v1dd' AND feature_set_id = 'v1dd_soma_features'",), rows_written=11)

In [10]:
feature_set = CellFeatureSet(
    id=FEATURE_SET_ID,
    description="V1DD soma morphology and nucleus-derived features.",
    feature_definition_ids=[fd.id for fd in feature_defs],
    extraction_method="Loaded from v1dd_v1196 soma_features.parquet.",
    project_id=PROJECT_ID,
)
write_models([feature_set], output_root=OUTPUT_ROOT)

WrittenResult(class_name='CellFeatureSet', path=PosixPath('../results/v1dd_1196_v3/cellfeatureset'), mode='overwrite_scoped', predicates=("project_id = 'v1dd' AND id = 'v1dd_soma_features'",), rows_written=1)

In [11]:
wide_df = (
    soma_features_cc
    .select(["id"] + FEATURE_COLUMNS)
    .with_columns([pl.col(c).cast(pl.Float64) for c in FEATURE_COLUMNS])
    .to_pandas()
)
wide_df["project_id"] = PROJECT_ID
wide_df["feature_set_id"] = FEATURE_SET_ID

schema_wide = build_cell_feature_matrix_schema(feature_set, feature_defs, cell_index_column="id")
table_wide = pa.Table.from_pandas(wide_df, schema=schema_wide, preserve_index=False)
write_deltalake(
    OUTPUT_ROOT / f"cellfeatures/{FEATURE_SET_ID}",
    table_wide,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)

cfm = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FEATURE_SET_ID}",
    feature_set_id=FEATURE_SET_ID,
    parquet_path=f"file://{OUTPUT_ROOT.resolve()}/cellfeatures/{FEATURE_SET_ID}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
write_models([cfm], output_root=OUTPUT_ROOT)

WrittenResult(class_name='CellFeatureMatrix', path=PosixPath('../results/v1dd_1196_v3/cellfeaturematrix'), mode='overwrite_scoped', predicates=("project_id = 'v1dd' AND feature_set_id = 'v1dd_soma_features'",), rows_written=1)

In [12]:
print(f"wrote {len(feature_defs)} feature definitions")
print(f"wrote wide feature matrix rows: {table_wide.num_rows}")

wrote 11 feature definitions
wrote wide feature matrix rows: 121429
